# NER and Entity Resolution - spaCy + EntityRuler
This work is a part of the knowledge graph pipeline built for 26 Spring BigCo Studio at Cornell Tech, and belongs to Team 10.

The whole process of this part is run on Google Colab.

# 1. Set up GDrive

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
%cd /content/gdrive/MyDrive/NER_EntityResolution

/content/gdrive/MyDrive/NER_EntityResolution


# 2. Install required packages

In [3]:
!pip install -q pandas spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 88.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# 3. Preprocess data

In [4]:
!python src/preprocess.py \
  --input data/raw/stage1_records.csv \
  --cleaned-output data/processed/cleaned_input_stage1.csv \
  --invalid-output data/processed/invalid_rows_stage1.csv

[preprocess] Wrote 80 valid row(s) to: data/processed/cleaned_input_stage1.csv
[preprocess] Wrote 0 invalid row(s) to: data/processed/invalid_rows_stage1.csv


In [5]:
!ls data/processed/

cleaned_input_mock1.csv  cleaned_input_stage1.csv  invalid_rows_mock2.csv
cleaned_input_mock2.csv  invalid_rows_mock1.csv    invalid_rows_stage1.csv


In [6]:
import pandas as pd

cleaned_df = pd.read_csv("data/processed/cleaned_input_stage1.csv")
invalid_df = pd.read_csv("data/processed/invalid_rows_stage1.csv")

print("Cleaned input:")
display(cleaned_df)

print("Invalid rows:")
display(invalid_df)

Cleaned input:


,source_id,source_type,raw_text,source_url,date,company_seed,source_type_cleaned,raw_text_cleaned,source_url_cleaned,date_cleaned,company_seed_cleaned,company_seed_invalid,company_seed_invalid_reason,row_invalid_reasons,row_valid_for_ner
0,1,PUBMED,Safety and Efficacy of the BNT162b2 mRNA Covid...,https://pubmed.ncbi.nlm.nih.gov/33301246/,Dec-20,Pfizer,PUBMED,Safety and Efficacy of the BNT162b2 mRNA Covid...,https://pubmed.ncbi.nlm.nih.gov/33301246/,Dec-20,Pfizer,False,NaN,NaN,True
1,2,PUBMED,"Encorafenib, Cetuximab, and mFOLFOX6 in First-...",https://pubmed.ncbi.nlm.nih.gov/40444708/,Jun-25,Pfizer,PUBMED,"Encorafenib, Cetuximab, and mFOLFOX6 in First-...",https://pubmed.ncbi.nlm.nih.gov/40444708/,Jun-25,Pfizer,False,NaN,NaN,True
2,3,PUBMED,Tafamidis Treatment for Patients with Transthy...,https://pubmed.ncbi.nlm.nih.gov/30145929/,Sep-18,Pfizer,PUBMED,Tafamidis Treatment for Patients with Transthy...,https://pubmed.ncbi.nlm.nih.gov/30145929/,Sep-18,Pfizer,False,NaN,NaN,True
3,4,PUBMED,Palbociclib and Letrozole in Advanced Breast C...,https://pubmed.ncbi.nlm.nih.gov/27959613/,Nov-16,Pfizer,PUBMED,Palbociclib and Letrozole in Advanced Breast C...,https://pubmed.ncbi.nlm.nih.gov/27959613/,Nov-16,Pfizer,False,NaN,NaN,True
4,5,PUBMED,Ponsegromab for the Treatment of Cancer Cachex...,https://pubmed.ncbi.nlm.nih.gov/39282907/,Dec-24,Pfizer,PUBMED,Ponsegromab for the Treatment of Cancer Cachex...,https://pubmed.ncbi.nlm.nih.gov/39282907/,Dec-24,Pfizer,False,NaN,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,76,PUBMED,"uniQure, CSL Behring Launch $2B Hemophilia B P...",https://pubmed.ncbi.nlm.nih.gov/34143661/,Jun-21,CSL Behring,PUBMED,"uniQure, CSL Behring Launch $2B Hemophilia B P...",https://pubmed.ncbi.nlm.nih.gov/34143661/,Jun-21,CSL Behring,False,NaN,NaN,True
76,77,PUBMED,Outcomes of a patient support programme for su...,https://pubmed.ncbi.nlm.nih.gov/39291857/,Nov-24,CSL Behring,PUBMED,Outcomes of a patient support programme for su...,https://pubmed.ncbi.nlm.nih.gov/39291857/,Nov-24,CSL Behring,False,NaN,NaN,True
77,78,PUBMED,Clinical experience of switching patients with...,https://pubmed.ncbi.nlm.nih.gov/36170120/,Feb-23,CSL Behring,PUBMED,Clinical experience of switching patients with...,https://pubmed.ncbi.nlm.nih.gov/36170120/,Feb-23,CSL Behring,False,NaN,NaN,True
78,79,PUBMED,IL-6 inhibition with clazakizumab in patients ...,https://pubmed.ncbi.nlm.nih.gov/38796655/,Aug-24,CSL Behring,PUBMED,IL-6 inhibition with clazakizumab in patients ...,https://pubmed.ncbi.nlm.nih.gov/38796655/,Aug-24,CSL Behring,False,NaN,NaN,True


Invalid rows:


,source_id,source_type,raw_text,source_url,date,company_seed,source_type_cleaned,raw_text_cleaned,source_url_cleaned,date_cleaned,company_seed_cleaned,company_seed_invalid,company_seed_invalid_reason,row_invalid_reasons,row_valid_for_ner


# 4. Run spaCy + EntityRuler for NER

In [7]:
!python src/ner_spacy_entityruler.py \
  --input data/processed/cleaned_input_stage1.csv \
  --output outputs/ner/ner_results_stage1_entityruler.csv \
  --spacy-model en_core_web_sm

[ner_spacy_entityruler] Wrote 856 mention(s) to: outputs/ner/ner_results_stage1_entityruler.csv


In [8]:
ner_df = pd.read_csv("outputs/ner/ner_results_stage1_entityruler.csv")
display(ner_df)

,source_id,source_type,source_type_cleaned,source_url,source_url_cleaned,date,date_cleaned,company_seed,company_seed_cleaned,raw_text,raw_text_cleaned,raw_mention,entity_label,start_char,end_char,mention_source,mention_confidence
0,1,PUBMED,PUBMED,https://pubmed.ncbi.nlm.nih.gov/33301246/,https://pubmed.ncbi.nlm.nih.gov/33301246/,Dec-20,Dec-20,Pfizer,Pfizer,Safety and Efficacy of the BNT162b2 mRNA Covid...,Safety and Efficacy of the BNT162b2 mRNA Covid...,Pfizer,ORG,2000,2006,spacy_or_entityruler,NaN
1,2,PUBMED,PUBMED,https://pubmed.ncbi.nlm.nih.gov/40444708/,https://pubmed.ncbi.nlm.nih.gov/40444708/,Jun-25,Jun-25,Pfizer,Pfizer,"Encorafenib, Cetuximab, and mFOLFOX6 in First-...","Encorafenib, Cetuximab, and mFOLFOX6 in First-...",EC,ORG,94,96,spacy_or_entityruler,NaN
2,2,PUBMED,PUBMED,https://pubmed.ncbi.nlm.nih.gov/40444708/,https://pubmed.ncbi.nlm.nih.gov/40444708/,Jun-25,Jun-25,Pfizer,Pfizer,"Encorafenib, Cetuximab, and mFOLFOX6 in First-...","Encorafenib, Cetuximab, and mFOLFOX6 in First-...",Food and Drug Administration,ORG,630,658,spacy_or_entityruler,NaN
3,2,PUBMED,PUBMED,https://pubmed.ncbi.nlm.nih.gov/40444708/,https://pubmed.ncbi.nlm.nih.gov/40444708/,Jun-25,Jun-25,Pfizer,Pfizer,"Encorafenib, Cetuximab, and mFOLFOX6 in First-...","Encorafenib, Cetuximab, and mFOLFOX6 in First-...",EC,ORG,1038,1040,spacy_or_entityruler,NaN
4,2,PUBMED,PUBMED,https://pubmed.ncbi.nlm.nih.gov/40444708/,https://pubmed.ncbi.nlm.nih.gov/40444708/,Jun-25,Jun-25,Pfizer,Pfizer,"Encorafenib, Cetuximab, and mFOLFOX6 in First-...","Encorafenib, Cetuximab, and mFOLFOX6 in First-...",EC+mFOLFOX6,ORG,1231,1242,spacy_or_entityruler,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
851,79,PUBMED,PUBMED,https://pubmed.ncbi.nlm.nih.gov/38796655/,https://pubmed.ncbi.nlm.nih.gov/38796655/,Aug-24,Aug-24,CSL Behring,CSL Behring,IL-6 inhibition with clazakizumab in patients ...,IL-6 inhibition with clazakizumab in patients ...,A2,ORG,1070,1072,spacy_or_entityruler,NaN
852,80,PUBMED,PUBMED,https://pubmed.ncbi.nlm.nih.gov/31404703/,https://pubmed.ncbi.nlm.nih.gov/31404703/,Oct-19,Oct-19,CSL Behring,CSL Behring,Next-generation Fc receptor-targeting biologic...,Next-generation Fc receptor-targeting biologic...,CSL777,ORG,682,688,spacy_or_entityruler,NaN
853,80,PUBMED,PUBMED,https://pubmed.ncbi.nlm.nih.gov/31404703/,https://pubmed.ncbi.nlm.nih.gov/31404703/,Oct-19,Oct-19,CSL Behring,CSL Behring,Next-generation Fc receptor-targeting biologic...,Next-generation Fc receptor-targeting biologic...,Pan Fc Receptor Interacting Molecule,ORG,693,729,spacy_or_entityruler,NaN
854,80,PUBMED,PUBMED,https://pubmed.ncbi.nlm.nih.gov/31404703/,https://pubmed.ncbi.nlm.nih.gov/31404703/,Oct-19,Oct-19,CSL Behring,CSL Behring,Next-generation Fc receptor-targeting biologic...,Next-generation Fc receptor-targeting biologic...,FcRn,ORG,799,803,spacy_or_entityruler,NaN


# 5. Run alias resolution

In [9]:
!python src/resolve_alias.py \
  --input outputs/ner/ner_results_stage1_entityruler.csv \
  --resolved-output outputs/entity_resolution/resolved_entities_stage1_entityruler.csv \
  --review-output outputs/entity_resolution/review_queue_stage1_entityruler.csv

[resolve_alias] Wrote 856 resolved row(s) to: outputs/entity_resolution/resolved_entities_stage1_entityruler.csv
[resolve_alias] Wrote 33 review pair(s) to: outputs/entity_resolution/review_queue_stage1_entityruler.csv


In [10]:
import os

for path in [
    "outputs/entity_resolution/resolved_entities_stage1_entityruler.csv",
    "outputs/entity_resolution/review_queue_stage1_entityruler.csv",
]:
    print(path)
    print("exists:", os.path.exists(path))
    if os.path.exists(path):
        print("size:", os.path.getsize(path), "bytes")
    print("-" * 40)

outputs/entity_resolution/resolved_entities_stage1_entityruler.csv
exists: True
size: 4071152 bytes
----------------------------------------
outputs/entity_resolution/review_queue_stage1_entityruler.csv
exists: True
size: 6051 bytes
----------------------------------------


# 6. Candidate relations

In [11]:
import pandas as pd

resolved_df = pd.read_csv("outputs/entity_resolution/resolved_entities_stage1_entityruler.csv")
display(resolved_df)

,source_id,source_type,source_url,date,company_seed,raw_text,raw_mention,entity_label,start_char,end_char,normalized_name,base_name,legal_suffixes,canonical_group_id,canonical_name,merge_confidence,merge_decision,evidence_note
0,1,PUBMED,https://pubmed.ncbi.nlm.nih.gov/33301246/,Dec-20,Pfizer,Safety and Efficacy of the BNT162b2 mRNA Covid...,Pfizer,ORG,2000,2006,PFIZER,PFIZER,NaN,67,Pfizer,0.88,auto_merge,auto_merge_component
1,2,PUBMED,https://pubmed.ncbi.nlm.nih.gov/40444708/,Jun-25,Pfizer,"Encorafenib, Cetuximab, and mFOLFOX6 in First-...",EC,ORG,94,96,EC,EC,NaN,94,EC,1.00,singleton,singleton_no_merge_needed
2,2,PUBMED,https://pubmed.ncbi.nlm.nih.gov/40444708/,Jun-25,Pfizer,"Encorafenib, Cetuximab, and mFOLFOX6 in First-...",Food and Drug Administration,ORG,630,658,FOOD AND DRUG ADMINISTRATION,FOOD AND DRUG ADMINISTRATION,NaN,100,Food and Drug Administration,0.60,review_needed,very_high_string_similarity_but_not_exact
3,2,PUBMED,https://pubmed.ncbi.nlm.nih.gov/40444708/,Jun-25,Pfizer,"Encorafenib, Cetuximab, and mFOLFOX6 in First-...",EC,ORG,1038,1040,EC,EC,NaN,94,EC,1.00,singleton,singleton_no_merge_needed
4,2,PUBMED,https://pubmed.ncbi.nlm.nih.gov/40444708/,Jun-25,Pfizer,"Encorafenib, Cetuximab, and mFOLFOX6 in First-...",EC+mFOLFOX6,ORG,1231,1242,EC MFOLFOX6,EC MFOLFOX6,NaN,252,EC+mFOLFOX6,1.00,singleton,singleton_no_merge_needed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
851,79,PUBMED,https://pubmed.ncbi.nlm.nih.gov/38796655/,Aug-24,CSL Behring,IL-6 inhibition with clazakizumab in patients ...,A2,ORG,1070,1072,A2,A2,NaN,137,A2,1.00,singleton,singleton_no_merge_needed
852,80,PUBMED,https://pubmed.ncbi.nlm.nih.gov/31404703/,Oct-19,CSL Behring,Next-generation Fc receptor-targeting biologic...,CSL777,ORG,682,688,CSL777,CSL777,NaN,122,CSL777,1.00,singleton,singleton_no_merge_needed
853,80,PUBMED,https://pubmed.ncbi.nlm.nih.gov/31404703/,Oct-19,CSL Behring,Next-generation Fc receptor-targeting biologic...,Pan Fc Receptor Interacting Molecule,ORG,693,729,PAN FC RECEPTOR INTERACTING MOLECULE,PAN FC RECEPTOR INTERACTING MOLECULE,NaN,30,Pan Fc Receptor Interacting Molecule,1.00,singleton,singleton_no_merge_needed
854,80,PUBMED,https://pubmed.ncbi.nlm.nih.gov/31404703/,Oct-19,CSL Behring,Next-generation Fc receptor-targeting biologic...,FcRn,ORG,799,803,FCRN,FCRN,NaN,234,FcRn,1.00,singleton,singleton_no_merge_needed


In [12]:
!mkdir -p outputs/relations

!python src/extract_candidate_relations.py \
  --input outputs/entity_resolution/resolved_entities_stage1_entityruler.csv \
  --output outputs/relations/candidate_relations_stage1_entityruler.csv

[extract_candidate_relations] Wrote 11 candidate row(s) to: outputs/relations/candidate_relations_stage1_entityruler.csv
